In [1]:
# generate_configs_pm_new_both.py
import os
import yaml
from copy import deepcopy

SR = 16000
BLOCK = 64
SIGLEN = 64000
STEPS = 10000
SAVE_AUDIO_STEPS = [10, 100, 1000]

PM_VERSION = "new"
VARIATIONS = [
    "convolved_Bernardel_ear_norm",
    "convolved_Bernardel_ff_norm",
    "convolved_StradCopy_ear_norm",
    "convolved_StradCopy_ff_norm",
    "convolved_Sundin_ear_norm",
    "convolved_Sundin_ff_norm",
]
STRINGS = ["A", "D", "E", "G"]  # per-string (violin only)
FULL = "full"

# ARMA resonance configuration
AR_ORDER = 64
MA_ORDER = 64
RES_WINDOW_SIZE = 2048
RES_MAX_REFLECTION = 1.0

FAMILIES = ("conv", "arma64x64")  # generate both

def clean_var_name(var_full: str) -> str:
    return var_full.replace("convolved_", "")

def base_config(dataset_tag: str, data_location: str, preprocess_out: str, run_name: str):
    return {
        "train": {
            "name": run_name,
            "root": "runs/",
            "steps": STEPS,
            "batch_size": 8,
            "loss_window": 100,
            "save_audio_steps": SAVE_AUDIO_STEPS,
            "save_audio_num_comparisons": 3,
            "gradient_clip_norm": 3.0,
            "learning_rate": 0.001,
            "lr_decay_steps": 10000,
            "lr_decay_rate": 0.98,
            "lr_staircase": False,
            "svl_activation_threshold": 1000.0,
            "hdl_activation_threshold": 1000.0,  # harmless/unused
            "dataloader_num_workers": 4,
            "dataloader_pin_memory": True,
            "preprocessed_data_dir": "preprocessed",
        },
        "preprocess": {
            "out_dir": preprocess_out,
            "sampling_rate": SR,
            "block_size": BLOCK,
            "signal_length": SIGLEN,
            "oneshot": False,
            "crepe_model_path": "",
        },
        "data": {
            "data_location": data_location,
            "extension": "wav",
            "dataset_name": dataset_tag,
        },
        "model": {
            "sampling_rate": SR,
            "block_size": BLOCK,
            "signal_length": SIGLEN,
            "use_inharmonicity": False,
            "encoder": {
                "use_encoder": True,
                "rnn_channels": 512,
                "z_dims": 16,
                "z_time_steps": 125,
                "mfcc": {
                    "n_mfcc": 30,
                    "n_mels": 128,
                    "f_min": 20.0,
                    "f_max": 8000.0,
                    "log_mels": True,
                },
            },
            "decoder": {
                "n_harmonic": 60,
                "n_bands": 65,
                "rnn_channels": 512,
                "ch": 512,
                "layers_per_stack": 3,
                "noise_bias": -5.0,
            },
            "helmholtz": {
                "use_helmholtz_synthesis": False,
                "β_min": 0.05,
                "β_max": 0.50,
                "α_min": -1.0,
                "α_max": 1.0,
                "notch_width": 0.05,
                "n_residuals": 10,
                "residual_scale": 1.0,
                "inharmonicity_b_max": 0.0005,
            },
            # resonance gets filled per-family
            "use_room": False,
            "room_type": "ar",
            "room_window_size": 4096,
            "room_ar_order": 16,
            "room_add_dry": True,
            "room_max_reflection": 0.999,
            "room_normalization_type": "rms",
        },
        "loss": {
            "mssl": {"scales": [2048, 1024, 512, 256, 128, 64], "overlap": 0.75},
            "use_source_variance": False,
            "source_variance": {
                "weight": 0.0,
                "loudness_threshold": 0.2,
                "pitch_threshold": 50.0,
                "distribution_type": "var",
                "power_type": "pwr",
                "per_pitch": False,
                "normalize_metrics": True,
                "epsilon": 1e-8,
                "n_harmonics": 10,
            },
            "use_helmholtz_deviation_loss": False,
            "violin_physics": {
                "weight": 0.0,
                "smoothness_β": 0.0,
                "smoothness_α": 0.0,
                "smoothness_γ": 0.0,
                "smoothness_B": 0.0,
                "residual_magnitude": 1.0,
                "residual_loss_type": "l2",
                "use_activation_filter": True,
                "loudness_threshold": 0.2,
                "pitch_threshold": 20.0,
            },
        },
    }

def apply_resonance_family(cfg: dict, family: str):
    m = cfg["model"]; m["use_resonance"] = True
    if family == "conv":
        m.update({
            "resonance_type": "convolutional",
            "resonance_length": 2048,
            "resonance_add_dry": False,
            "resonance_mask_ir": False,
            "resonance_position": "before_noise",
            "resonance_normalization_type": "None",
        })
        for k in ("resonance_window_size","resonance_ar_order","resonance_ma_order","resonance_max_reflection"):
            m.pop(k, None)
    elif family == "arma64x64":
        m.update({
            "resonance_type": "arma",
            "resonance_window_size": RES_WINDOW_SIZE,
            "resonance_ar_order": AR_ORDER,
            "resonance_ma_order": MA_ORDER,
            "resonance_max_reflection": RES_MAX_REFLECTION,
            "resonance_add_dry": False,
            "resonance_position": "before_noise",
            "resonance_normalization_type": "None",
        })
        for k in ("resonance_length","resonance_mask_ir","resonance_psd_prior","resonance_n_fft","resonance_psd_ema_alpha"):
            m.pop(k, None)
    else:
        raise ValueError(f"Unknown family: {family}")

def write_config(cfg: dict, out_path: str):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

def build_entries_new_only():
    """Yield (dataset_tag, data_location, preprocess_out, var_name, split)."""
    for var_full in VARIATIONS:
        label = clean_var_name(var_full)
        # per-string
        for s in STRINGS:
            tag = f"physical_model/{PM_VERSION}/{var_full}/{s}"
            data_loc = f"dataset/physical_model/{PM_VERSION}/{var_full}/{s}"
            prep_out = f"preprocessed/physical_model/{PM_VERSION}/{var_full}/{s}"
            yield tag, data_loc, prep_out, label, s
        # full
        s = FULL
        tag = f"physical_model/{PM_VERSION}/{var_full}/{s}"
        # >>> FIX: include /full in BOTH paths <<<
        data_loc = f"dataset/physical_model/{PM_VERSION}/{var_full}/full"
        prep_out = f"preprocessed/physical_model/{PM_VERSION}/{var_full}/full"
        yield tag, data_loc, prep_out, label, s

def main():
    out_root = "configs"
    os.makedirs(out_root, exist_ok=True)
    total = 0

    for tag, data_loc, prep_out, var_name, split in build_entries_new_only():
        for family in FAMILIES:
            fam_suffix = "" if family == "conv" else "__arma64x64"
            base_name = f"{var_name}_{split}{fam_suffix}"
            cfg0 = base_config(tag, data_loc, prep_out, base_name)
            apply_resonance_family(cfg0, family)

            if split == FULL:
                # STD: SVL 0/1/10
                for w in [0.0, 1.0, 10.0]:
                    c = deepcopy(cfg0)
                    c["model"]["helmholtz"]["use_helmholtz_synthesis"] = False
                    c["loss"]["use_source_variance"] = (w > 0.0)
                    c["loss"]["source_variance"]["weight"] = float(w)
                    c["loss"]["use_helmholtz_deviation_loss"] = False
                    c["loss"]["violin_physics"]["weight"] = 0.0
                    run = f"{base_name}__std" if w == 0.0 else f"{base_name}__std_svl_w{int(w)}"
                    c["train"]["name"] = run
                    write_config(c, os.path.join(out_root, f"{run}.yaml")); total += 1
                # VIO: VPL 0/1/10
                for w in [0.0, 1.0, 10.0]:
                    c = deepcopy(cfg0)
                    c["model"]["helmholtz"]["use_helmholtz_synthesis"] = True
                    c["loss"]["use_source_variance"] = False
                    c["loss"]["source_variance"]["weight"] = 0.0
                    c["loss"]["use_helmholtz_deviation_loss"] = (w > 0.0)
                    c["loss"]["violin_physics"]["weight"] = float(w)
                    run = f"{base_name}__vio" if w == 0.0 else f"{base_name}__vio_vpl_w{int(w)}"
                    c["train"]["name"] = run
                    write_config(c, os.path.join(out_root, f"{run}.yaml")); total += 1
            else:
                # per-string: VPL 0/1/10
                for w in [0.0, 1.0, 10.0]:
                    c = deepcopy(cfg0)
                    c["model"]["helmholtz"]["use_helmholtz_synthesis"] = True
                    c["loss"]["use_source_variance"] = False
                    c["loss"]["source_variance"]["weight"] = 0.0
                    c["loss"]["use_helmholtz_deviation_loss"] = (w > 0.0)
                    c["loss"]["violin_physics"]["weight"] = float(w)
                    run = f"{base_name}__vio" if w == 0.0 else f"{base_name}__vio_vpl_w{int(w)}"
                    c["train"]["name"] = run
                    write_config(c, os.path.join(out_root, f"{run}.yaml")); total += 1

    print(f"Generated {total} configs in '{out_root}/'.")
    print("Examples:")
    print("  - configs/Bernardel_ear_norm_full__std.yaml")
    print("  - configs/Bernardel_ear_norm_full__vio_vpl_w10.yaml")
    print("  - configs/Bernardel_ear_norm_full__arma64x64__std.yaml")
    print("  - configs/Bernardel_ear_norm_A__arma64x64__vio_vpl_w10.yaml")

if __name__ == "__main__":
    main()


Generated 216 configs in 'configs/'.
Examples:
  - configs/Bernardel_ear_norm_full__std.yaml
  - configs/Bernardel_ear_norm_full__vio_vpl_w10.yaml
  - configs/Bernardel_ear_norm_full__arma64x64__std.yaml
  - configs/Bernardel_ear_norm_A__arma64x64__vio_vpl_w10.yaml
